# 8. 提升方法

## 引入概念：GBDT

**GBDT** 全称 **Gradient Boosting Decision Tree**，梯度提升决策树，属于 Boosting 串行集成学习算法，基学习器是 CART 回归树。

**含义**：一棵一棵树依次串行训练，每棵新树去拟合上一轮模型损失函数的负梯度（伪残差），最后把所有树预测值累加起来得到最终结果。

**理解**：第一棵树先粗略预测，第二棵专门修正第一棵的误差，第三棵修正前两棵合起来的误差，不断补坑。


## 1. 引言

回归：随机森林，它是装袋 Bagging 思想：一次性同时训练一大堆比较强、但是不稳定的大树，每棵树自己独立学、互不打扰，最后大家一起投票取平均。（**并行**）

例如：随机森林 = 班里几十个中等水平的同学，同时独立做题，最后答案一起投票，抵消各自的失误。

现在换一条完全相反的思路：如果我手里没有这些中等水平的同学，只有很多很多水平很差的学生 —— 他们只比瞎蒙强一点点，单独一个人做题正确率很低，能不能把这群 "差生" 组合起来，变成一个很厉害的高手？

这个就需要使用：**Boosting，提升方法**。


## 2. 概念

PAC 学习、弱可学习 → 强可学习

很早的时候学界就在争论这个问题：

- **弱学习器（弱模型）**：效果仅仅比随机瞎猜好一点点的模型
- **强学习器（强模型）**：可以做到误差非常小、预测很准的模型

科学家提出猜想：如果我只有弱模型，能不能拼出强模型？**1990 年 Schapire 证明了：可以！**

他最早的思路特别好懂，就是 "三个臭皮匠" 版本：

1. 先训练第一个弱模型 A，它会做错一部分样本；
2. 专门挑 A 做错的那些题目，训练第二个弱模型 B，让 B 专门搞定 A 不会的题；
3. 再训练第三个弱模型 C，专门用来判断：A 和 B 答案不一样的时候，谁说的对；最后三个一起投票，整体准确率就明显高于单独任何一个。

而且这个思路可以不断叠加上去，不断新增弱模型、持续补错，误差可以无限压低。"Boosting（提升）" 这个名字就是这么来的 —— 把弱能力 "提升" 成强能力。

**记住 Boosting 的核心规则：串行排队训练，后面的模型，专门去弥补前面所有模型犯的错误。**

**Boosting 与随机森林对比：**

- 随机森林：并行，树之间互不依赖，可以一起训练；
- Boosting：串行，必须等上一个模型训练完，算出哪里错了，下一个才知道要学什么，不能并行。


## 3. 一个树桩有多弱

提升里最常用的弱模型是**树桩（decision stump）**：只切一刀的决策树。

案例：比较树桩、深度 3 的树和长满的树的准确率。


In [1]:
# 准备数据
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# 定义函数
def make_2d(seed=273, n=800):
    # 固定随机种子，保证每次运行生成一样的数据
    r = np.random.default_rng(seed)
    # 月收入：3千~25千，保留1位小数
    inc = np.round(r.uniform(3, 25, n), 1)
    # 负债收入比，限制在0.01~0.99之间
    debt = np.round(np.clip(r.beta(2.2, 3, n), 0.01, 0.99), 3)
    
    # 真实规则：负债 > 0.22 + 0.022*收入， 或者收入<6千同时负债>0.2 → 属于高风险
    risky = (debt > 0.22 + 0.022 * inc) | ((inc < 6) & (debt > 0.2))
    
    # 随机挑12%样本翻转标签，制造噪声：^ 是异或，满足risky，但抽到噪声样本，标签反过来
    y = (risky ^ (r.random(n) < 0.12)).astype(int)
    
    # 把收入、负债两个特征拼在一起，形成两列特征矩阵
    X = np.c_[inc, debt]
    
    # 划分训练、验证
    return X[:200], y[:200], X[200:], y[200:]

# 生成数据
X2, y2, X2v, y2v = make_2d()

# 计算基准线：全猜多数类的准确率
print("全猜多数类：", round(max(y2v.mean(), 1 - y2v.mean()), 4))

# 训练并对比不同深度的决策树
for name, depth in [('一个树桩', 1), ("深度3的树", 3), ("长满的树", None)]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X2, y2)
    print(f"{name}: 训练准确率 {m.score(X2, y2):.4f}, 验证准确率 {m.score(X2v, y2v):.4f}")

全猜多数类： 0.5983
一个树桩: 训练准确率 0.7250, 验证准确率 0.7083
深度3的树: 训练准确率 0.8250, 验证准确率 0.8117
长满的树: 训练准确率 1.0000, 验证准确率 0.7067


一个树桩的验证准确率 0.708，比全猜多数类（0.598）强一些，可也只强这么一点：它只能沿着一个特征切一刀，斜着走的边界完全画不出来。它**偏差大、方差小**：换几个人重新训练，那一刀大概还在差不多的位置。正好和第 6 讲长满的树相反。随机森林用平均压方差；提升要做的，是把一个个偏差大的树桩，一步步组合成偏差小的模型。


## 4. 如何进行一轮一轮的训练

补充：树桩就是 Boosting（AdaBoost 等提升算法）最经典的弱学习器。

前面：AdaBoost 就是一轮一轮训练树桩（只切一刀的弱分类器），后一个树桩专门去搞定前一个树桩分错的样本。

核心手段：给每个样本带上权重 $w$：

- 权重越大，代表这个样本越重要；
- 训练下一个树桩的时候，优先保证把权重高的样本分对；
- 第一轮：所有样本权重一模一样，大家同等重要；
- 衡量好坏不再是 "分错几个"，而是加权错误 $\varepsilon$ = 所有分错样本的权重加起来。

标签用 +1（违约）、-1（正常），不用 0、1，是为了后面 AdaBoost 公式写起来简洁，这是 AdaBoost 的惯例。

AdaBoost 的回答是样本权重：每个人有一个权重，加起来是 1；找一刀时，看的是加权错误（判错的人的权重之和）。例如：10 个人（两个特征 $x_1$、$x_2$，1 表示违约、−1 表示正常）。

| 样本 | x1 | x2 | y | x1 ≤ 5.5？ | 树桩判断 h | 是否判错 | 权重 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 2 | 8 | +1 | 是 | −1 | 错 | 0.1 |
| 2 | 9 | 8 | +1 | 否 | +1 | 对 | 0.1 |
| 3 | 4 | 4 | −1 | 是 | −1 | 对 | 0.1 |
| 4 | 9 | 1 | +1 | 否 | +1 | 对 | 0.1 |
| 5 | 3 | 6 | −1 | 是 | −1 | 对 | 0.1 |
| 6 | 6 | 6 | +1 | 否 | +1 | 对 | 0.1 |
| 7 | 8 | 5 | −1 | 否 | +1 | 错 | 0.1 |
| 8 | 1 | 7 | −1 | 是 | −1 | 对 | 0.1 |
| 9 | 5 | 5 | −1 | 是 | −1 | 对 | 0.1 |
| 10 | 4 | 9 | −1 | 是 | −1 | 对 | 0.1 |

1 号和 7 号判错，加权错误 `ε = 0.1 + 0.1 = 0.2`，下一轮把这两个人的权重调大。

`best_stump(X, y, w)` 遍历每个特征、以相邻取值的中点为候选阈值，并试两个方向 `s = ±1`；预测 `h = np.where(X[:, j] <= t, s, -s)`，用 `err = w[h != y].sum()` 计算加权错误，返回误差最小的那一刀。

In [2]:
# 目的：一共10个人，2个特征，第一轮所有人权重都是0.1
# 遍历所有可能的“一刀切”，找到【加权错误最小】的那一刀，也就是最优树桩
import numpy as np

# X10：10行2列，10个样本，两列分别是特征x1、x2
X10 = np.array([[2, 8], [9, 8], [4, 4], [9, 1], [3, 6], [6, 6], [8, 5], [1, 7], [5, 5], [4, 9]])
# y10：标签，1代表违约，-1代表正常，AdaBoost经典±1标记
y10 = np.array([1, 1, -1, 1, -1, 1, -1, -1, -1, -1])
# w：权重数组，一共10个样本，初始权重全部等于0.1；所有权重相加总和=1
w = np.full(10, 0.1)

def best_stump(X, y, w):
    """
    遍历每一个特征、每一个阈值、两种判定方向
    返回：加权误差最小的那一刀（最优树桩）
    返回内容：(最小加权错误, 第几号特征, 阈值, 判定方向s, 这一刀对全部样本的预测结果h)
    """
    best = None  # 用来保存目前找到最好的一刀，初始为空
    # 循环：遍历每一列特征 j，j=0是x1，j=1是x2
    for j in range(X.shape[1]):
        # 取出当前特征这一列所有的值，去重，得到该特征所有不同取值
        vals = np.unique(X[:, j])
        # 相邻两个取值中点作为候选阈值，这是决策树找分割点的标准做法
        # 比如取值是 [1,3,4]，中点就是 (1+3)/2=2、(3+4)/2=3.5
        for t in (vals[:-1] + vals[1:]) / 2:
            # s 代表判定方向：两种选择
            # s=1：如果样本特征 ≤阈值 → 判为+1（违约）
            # s=-1：如果样本特征 >阈值 → 判为-1（正常）
            for s in [1, -1]:
                # h：当前这一刀对全部样本的预测结果数组
                # np.where：满足 X[:,j] ≤ t → 输出s；否则输出-s
                h = np.where(X[:, j] <= t, s, -s)
                # 计算加权错误：把【预测h不等于真实标签y】的那些样本的权重加起来
                err = w[h != y].sum()
                # 如果还没有最优，或者当前这一刀误差比已经存的最优更小 → 更新最优
                # 减 1e-12 是浮点数精度容错，避免浮点相等带来的bug
                if best is None or err < best[0] - 1e-12:
                    best = (err, j, t, s, h)
    return best

# 调用函数，拿到最优树桩的全部信息
eps, j, t, s, h = best_stump(X10, y10, w)
# 打印结果，给人看懂：j从0开始，所以特征编号+1
print(f"最好的一刀：特征 x{j + 1} ≤ {t} 判{'违约' if s == 1 else '正常'}，否则判{'正常' if s == 1 else '违约'}")
print("这一刀的判断：", h.tolist())
# 找到所有预测错误的样本下标（从1开始编号给学生看）
wrong_list = [k + 1 for k in range(10) if h[k] != y10[k]]
print(f"判错的人：{wrong_list}   加权错误 ε = {round(eps, 4)}")

最好的一刀：特征 x1 ≤ 5.5 判正常，否则判违约
这一刀的判断： [-1, 1, -1, 1, -1, 1, 1, -1, -1, -1]
判错的人：[1, 7]   加权错误 ε = 0.2


## 5. 发言权 α 和新的权重

AdaBoost：不断训练弱分类器（简单一刀），给每个弱分类器分配发言权 $\alpha$，同时调整样本权重，让下一个分类器重点关注上一轮分错的样本。

例子：一共 10 个样本，这个弱分类器猜对 8 个、猜错 2 个，错误率 $\varepsilon = 2/10 = 0.2$。

这一刀判对了 80%，比瞎猜（50%）强。强多少，就给它多大的发言权：

$$\alpha = \frac{1}{2}\ln\frac{1-\varepsilon}{\varepsilon}$$

$\varepsilon$ 是本轮错误率：

- 分类器很准（$\varepsilon$ 很小）→ $\alpha$ 很大，投票话语权很重；
- 分类器很差（$\varepsilon$ 接近 0.5，和瞎蒙一样）→ $\alpha \approx 0$，几乎没有发言权；
- 分类器比瞎猜还差（$\varepsilon > 0.5$）→ $\alpha$ 是负数，相当于反过来用它的结果。

这个例子里 $\varepsilon = 0.2$，算出 $\alpha = 0.5 \times \ln(4) \approx 0.693$。

**更新样本权重：错的加重、对的减轻**

然后改权重：判错的人乘 $e^{\alpha}$（变重），判对的人乘 $e^{-\alpha}$（变轻），再除以总和、重新凑成 1。用 $y_i \cdot h(x_i)$ 判断对错（判对是 1，判错是 −1），一行就能写完：

$$w \leftarrow w \cdot e^{-\alpha \cdot y_i h(x_i)}$$

- 猜对的时候：$y_i h(x_i) = 1$，乘 $e^{-\alpha}$ → 权重变小
- 猜错的时候：$y_i h(x_i) = -1$，乘 $e^{\alpha}$ → 权重变大

$Z$ 是归一化因子：全部权重求和，除以 $Z$，保证所有权重加起来 = 1（概率分布）。

> AdaBoost 最巧妙的设计：抹平当前分类器的能力，强制下一轮弱学习器去学习剩下的、没搞定的样本。

**说明**：下面的单元格用 `y10_h` 表示 `y * h`，但数组里实际写了 6 个 −1（正好是上一格的预测 `h`），而不是 `y10 * h` 得到的 2 个 −1。因此运行后“判错的人的权重合计”是 0.8571。严格按上面表格（1 号和 7 号判错）计算，`y10 * h = [-1, 1, 1, 1, 1, 1, -1, 1, 1, 1]`，两个判错样本的权重从 0.1 升到 0.25，其余 8 人降到 0.0625，判错权重合计为 0.5。

## 6. 三轮合起来：按发言权投票

前面：找一刀（弱分类器）→ 算发言权 $\alpha$ → 修改样本权重，让错的样本变重要。

重复 "找一刀 → 算 $\alpha$ → 改权重"，三轮以后，把三刀的判断按发言权加起来，看正负；重复多轮，最后把所有弱分类器按发言权加权投票，拼成一个很强的模型！


In [3]:
# 导入numpy，用来做向量、指数、对数计算
import numpy as np
# 1、设置本轮弱分类器的错误率eps --------
eps = 0.2
# 例子：10个样本，错2个，错误率=2/10=0.2
# 2、计算这个弱分类器的发言权 alpha --------
# alpha = 0.5 * ln( (1-eps)/eps )，AdaBoost标准公式
alpha = 0.5 * np.log((1 - eps) / eps)
# 准备标记y10：标记每个样本本轮是猜对还是猜错
# y10 * h = 1 代表猜对；y10 * h = -1 代表猜错
# 这里：10个样本，2个猜错，8个猜对
# 我们写成 y10*h 的结果，2个-1，8个1
y10_h = np.array([-1, 1, -1, 1, -1, 1, 1, -1, -1, -1])
# 4、初始权重：10个样本一开始权重均等，每个0.1
w = np.ones(10) / 10
# 5、权重更新：乘上系数 e^(-alpha * y10_h)，错的乘e^α，对的乘e^(-α)
w_new = w * np.exp(-alpha * y10_h)
# 打印中间结果
print("α =", round(alpha, 4), "  e^α =", round(np.exp(alpha), 4), "  e^(-α) =", round(np.exp(-alpha), 4))
print("乘完指数以后（还没归一化）：", np.round(w_new, 4).tolist(), "  合计", round(w_new.sum(), 4))
# 归一化：除以总和Z，让所有权重之和=1
w_new = w_new / w_new.sum()
# 打印归一之后的权重
print("归一化后：", np.round(w_new, 4).tolist())
# 本轮猜错的那些样本
# y10_h == -1 就是猜错的样本
print("判错的人的权重合计：", round(w_new[y10_h != 1].sum(), 4))


α = 0.6931   e^α = 2.0   e^(-α) = 0.5
乘完指数以后（还没归一化）： [0.2, 0.05, 0.2, 0.05, 0.2, 0.05, 0.05, 0.2, 0.2, 0.2]   合计 1.4
归一化后： [0.1429, 0.0357, 0.1429, 0.0357, 0.1429, 0.0357, 0.0357, 0.1429, 0.1429, 0.1429]
判错的人的权重合计： 0.8571


- $h_m(x)$：第 m 把 "刀"，它的输出只有两种：+1、-1（比如 +1 代表判违约，-1 代表正常）
- $\alpha_m$：这把刀的发言权，准确率越高，$\alpha_m$ 越大，投票分量越重
- $F(x) = \sum \alpha_m h_m(x)$：加权总分，把每一把刀的判断 × 它的发言权，全部加起来
- $\text{sign}(F(x))$：符号函数。总分 > 0 → 最终判定 +1；总分 < 0 → 最终判定 -1

核心思想：每一轮只需要一个 "比瞎猜好一点点" 的弱分类器（$\varepsilon < 0.5$），多轮加权投票，强强联合，最后整体正确率很高。

α 随 ε 的变化：

1. $\varepsilon$ 越小（本轮弱分类器错得越少）→ $\alpha$ 越大，投票话语权越强；
2. $\varepsilon = 0.5$（和抛硬币瞎猜一样）→ $\alpha = 0$，这把刀直接没有发言权，等于不用它；
3. $\varepsilon > 0.5$（这个分类器还不如瞎猜，反向输出才准）→ $\alpha$ 是负数，相当于把它的判断反过来用。

3 轮运行结果：第 1 轮 `x1 ≤ 5.5` 判正常，第 2 轮 `x2 ≤ 7.5` 判正常，第 3 轮又是 `x1 ≤ 5.5` 判正常；`F(x)` 的符号与真实标签相比，10 个里对 8 个。

In [4]:
# 目的：把“找一刀 → 算 α → 改权重”重复 3 轮，再按 α 加权投票
def adaboost(X, y, rounds):
    w = np.full(len(y), 1 / len(y)) # 初始化权重，全部分为 1/N
    model = []                      # 用来存储每一轮的 (alpha, 特征, 阈值, 方向)
    
    for m in range(rounds):
        # 1. 找当前权重下，加权误差最小的树桩
        eps, j, t, s, h = best_stump(X, y, w)
        
        # 2. 计算这个树桩的话语权 alpha
        # eps 越小，alpha 越大，话语权越重
        alpha = 0.5 * np.log((1 - eps) / eps)
        
        # 3. 保存当前这棵树的参数
        model.append((alpha, j, t, s))
        
        # 4. 核心：更新样本权重（错题本机制）
        # 如果预测正确（y*h=1），权重乘 exp(-alpha) < 1，权重变小；
        # 如果预测错误（y*h=-1），权重乘 exp(alpha) > 1，权重变大。
        w = w * np.exp(-alpha * y * h)
        # 归一化，保证权重总和为 1
        w = w / w.sum()
        
    return model

def ada_score(model, X):
    """F(x) = Σ α·h(x)：正的判违约，负的判正常"""
    # 将所有树桩的预测结果（±1）乘以它们的话语权 alpha，然后累加
    return sum(a * np.where(X[:, j] <= t, s, -s) for a, j, t, s in model)

# 调用函数，跑 3 轮
model3 = adaboost(X10, y10, 3)

# 打印每一轮选出的弱分类器及其权重
for m, (a, j, t, s) in enumerate(model3, 1):
    print(f"第 {m} 轮：x{j + 1} ≤ {t} 判{'违约' if s == 1 else '正常'}，α = {a:.4f}")

# 获取最终的集成得分 F
F = ada_score(model3, X10)
print("F(x) =", np.round(F, 3).tolist())
print("sign(F) 和真实结果一样的人数：", int((np.sign(F) == y10).sum()), "/ 10")

第 1 轮：x1 ≤ 5.5 判正常，α = 0.6931
第 2 轮：x2 ≤ 7.5 判正常，α = 0.7332
第 3 轮：x1 ≤ 5.5 判正常，α = 0.4055
F(x) = [-0.365, 1.832, -1.832, 0.365, -1.832, 0.365, 0.365, -1.832, -1.832, -0.365]
sign(F) 和真实结果一样的人数： 8 / 10


## 7. sklearn 中 AdaBoost 使用

前面我们自己手写实现了 AdaBoost，现在：调用 sklearn 库里现成的 AdaBoost 分类器，和咱们手写版本做对照、找差别。

先抓住最重要的知识点：

1. 基础弱分类器：`DecisionTreeClassifier(max_depth=1)`，就是决策树桩（stump）—— 只能切一刀。
2. `n_estimators=3`：一共训练 3 个弱分类器（3 轮）。

### 7.1 案例实现

sklearn 默认使用 SAMME，`estimator_weights_` 里的权重不带手写公式前面的 `1/2`，所以数值是手写 α 的 2 倍；`estimator_errors_` 是每一轮的加权错误，`tree_.feature` 与 `tree_.threshold` 可以查看树桩用了哪个特征、切在哪里。

In [5]:
# 导入sklearn集成模块里的AdaBoost分类器
from sklearn.ensemble import AdaBoostClassifier
# 导入单层决策树（树桩），作为AdaBoost里面的弱学习器
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# 准备数据
X10 = np.array([[1,7],[2,8],[3,4],[9,1],[3,6],[4,5],[8,5],[1,8],[2,6],[4,9]])
y10 = np.array([1,1,-1,1,-1,-1,1,1,-1,-1])

# 初始化并训练AdaBoost
# estimator：弱分类器，max_depth=1 代表只能切一刀（决策树桩stump）
# n_estimators=3：一共训练3个弱分类器，也就是3轮
# random_state=0：固定随机种子，保证每次运行结果不变
ada3 = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=3,
    random_state=0
).fit(X10, y10)

# 循环遍历每一轮训练出来的弱分类器，从1开始编号
for m, tr in enumerate(ada3.estimators_, 1):
    # estimator_weights_：sklearn保存的每个弱分类器权重（SAMME，不带1/2）
    a = ada3.estimator_weights_[m - 1]
    # estimator_errors_：每一轮弱分类器的加权错误率ε
    eps = ada3.estimator_errors_[m - 1]
    # tr.tree_.feature[0]：这一刀切的是第几个特征；tr.tree_.threshold[0]：分割阈值
    print(f"第{m}轮：x{tr.tree_.feature[0]+1}<={tr.tree_.threshold[0]},加权错误 {eps:.4f},sklearn的权重 {a:.4f},除以2是 {a/2:.4f}")

# 用训练好的AdaBoost对10个样本进行预测
print("预测：", ada3.predict(X10).tolist())

第1轮：x1<=1.5,加权错误 0.3000,sklearn的权重 0.8473,除以2是 0.4236
第2轮：x1<=6.0,加权错误 0.3095,sklearn的权重 0.8023,除以2是 0.4012
第3轮：x2<=6.5,加权错误 0.2586,sklearn的权重 1.0531,除以2是 0.5266
预测： [1, -1, -1, 1, -1, -1, 1, 1, -1, -1]


## 8. 训练的轮数越多越好吗？

前面我们以为：AdaBoost 轮数加得越多，模型就越强、越准。

**核心问题：轮数越多，一定越好吗？**

AdaBoost 对噪声样本非常敏感！因为它会不断提高错分样本的权重；如果那个样本本身就是标错的噪声，权重越拉越高，后面一堆弱分类器专门围着这个噪声去拟合，最后把正常规律破坏掉。

所以轮数不是越大越好，存在一个最优轮数，到这个轮数就该停下。

### 8.1 案例

下面的代码训练 500 轮，用 `staged_predict` 依次查看前 1、2、…、500 个弱分类器组合的预测，对比训练集和验证集的准确率。

In [6]:
# 导入sklearn集成模块里的AdaBoost分类器
from sklearn.ensemble import AdaBoostClassifier
# 导入单层决策树（树桩），作为AdaBoost里面的弱学习器
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# 定义函数
def make_2d(seed=273, n=800):
    # 固定随机种子，保证每次运行生成一样的数据
    r = np.random.default_rng(seed)
    # 月收入：3千~25千，保留1位小数
    inc = np.round(r.uniform(3, 25, n), 1)
    # 负债收入比，限制在0.01~0.99之间
    debt = np.round(np.clip(r.beta(2.2, 3, n), 0.01, 0.99), 3)
    
    # 真实规则：负债 > 0.22 + 0.022*收入， 或者收入<6千同时负债>0.2 → 属于高风险
    risky = (debt > 0.22 + 0.022 * inc) | ((inc < 6) & (debt > 0.2))
    
    # 随机挑12%样本翻转标签，制造噪声
    y = (risky ^ (r.random(n) < 0.12)).astype(int)
    
    # 把收入、负债两个特征拼在一起，形成两列特征矩阵
    X = np.c_[inc, debt]
    
    # 划分训练、验证
    return X[:200], y[:200], X[200:], y[200:]

# 初始化数据
X2, y2, X2v, y2v = make_2d()

# 训练AdaBoost，一共500轮，弱分类器还是一刀树桩
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=500,
    random_state=0
).fit(X2, y2)

# staged_predict: 依次返回【只用前1个、前2个、……、前500个弱分类器】的预测结果
# tr_acc：训练集准确率数组，长度500
tr_acc = [np.mean(p == y2) for p in ada.staged_predict(X2)]
# va_acc：验证集准确率数组，长度500
va_acc = [np.mean(p == y2v) for p in ada.staged_predict(X2v)]

# 挑选几个有代表性轮数打印结果：1,5,10,50,100,200,500轮
for m in [1, 5, 10, 50, 100, 200, 500]:
    # m-1是数组下标（从0开始）
    print(f"{m:>3}轮：训练{tr_acc[m-1]:.4f}，验证 {va_acc[m-1]:.4f}")

# np.argmax找到验证准确率最大值对应的下标，+1转成轮号
best_round = int(np.argmax(va_acc)) + 1
print(f"验证准确率最高：第 {best_round} 轮", round(max(va_acc), 4))

  1轮：训练0.7250，验证 0.7083
  5轮：训练0.8000，验证 0.8067
 10轮：训练0.7650，验证 0.7367
 50轮：训练0.8250，验证 0.7900
100轮：训练0.8300，验证 0.7550
200轮：训练0.8650，验证 0.7383
500轮：训练0.9050，验证 0.7300
验证准确率最高：第 16 轮 0.815


训练准确率整体上升（1 轮 0.725 → 500 轮 0.905），模型越来越会“背”训练样本；验证集：5 轮 0.8067 很高，10 轮就掉下去，16 轮到顶峰 0.815，之后一路下滑，500 轮只剩 0.73。

说明：超过 16 轮之后，增加弱分类器对新数据反而有害，过拟合。

所以，这就是**早停（early stopping）**思想：我们可以监控验证集效果，一旦验证集开始持续下降，就停止训练，不要继续加轮数了。


## 9. 信贷数据上的 AdaBoost

最后在 `data/信贷数据.csv` 上跑一遍：

- 数值特征：年龄、月收入、负债收入比、额度使用率、逾期次数、工作年限，直接使用；
- 类别特征：学历、住房、职业，做 One-Hot 编码；
- 按 `是否违约` 分层抽样，20% 作测试集；
- 弱分类器仍是树桩，`n_estimators=300`。

用 `staged_predict_proba` 输出每一轮的违约概率，再算 `roc_auc_score`。

| 轮数 | 测试集 AUC |
| --- | --- |
| 1 | 0.5000 |
| 10 | 0.7266 |
| 50 | 0.7400 |
| 100 | 0.7543 |
| 300 | 0.7537 |

1 轮时 AUC 为 0.5，相当于随机猜；10 轮后升到 0.7266，之后提升有限，最优轮数 m=97 时 AUC 为 0.7544（100 轮 0.7543），300 轮反而略降到 0.7537——同样说明轮数不是越多越好。

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

# 导入sklearn集成模块里的AdaBoost分类器
from sklearn.ensemble import AdaBoostClassifier
# 导入单层决策树（树桩），作为AdaBoost里面的弱学习器
from sklearn.tree import DecisionTreeClassifier

# 加载数据
df = pd.read_csv('../data/信贷数据.csv')

# 分开定义：数值型特征、类别型特征
num = ["年龄", "月收入", "负债收入比", "额度使用率", "逾期次数", "工作年限"]
cat = ["学历", "住房", "职业"]

# 划分训练集、测试集，测试集占20%
X_train, X_test, y_train, y_test = train_test_split(
    df[num+cat], df['是否违约'].to_numpy(), 
    test_size=0.2, stratify=df['是否违约'], random_state=42
)

# 预处理流水线：数值列原样保留（passthrough）；分类列做OneHot独热编码
enc = ColumnTransformer([
    ('num', "passthrough", num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
]).fit(X_train)

# 对训练、测试数据执行编码转换
E_train, E_test = enc.transform(X_train), enc.transform(X_test)

# 构建AdaBoost：弱学习器是深度=1的决策树（树桩，只能切一刀），最多300轮
ada_c = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=300,
    random_state=0
).fit(E_train, y_train)

# staged_predict_proba：依次返回【前1个、前2个……前300个弱分类器组合】的预测概率
# roc_auc_score 提取每个阶段的正类概率 p[:,1] 计算 AUC
auc = [roc_auc_score(y_test, p[:, 1]) for p in ada_c.staged_predict_proba(E_test)]

# 挑选几个关键轮数打印结果：1、10、50、100、300轮，保留4位小数
print("测试集 AUC:", {m: round(auc[m-1], 4) for m in [1, 10, 50, 100, 300]})

best_idx = np.argmax(auc)
best_m = best_idx + 1
best_auc = auc[best_idx]
print(f"最优轮数 m={best_m}, 对应测试集AUC={best_auc:.4f}")

测试集 AUC: {1: 0.5, 10: 0.7266, 50: 0.74, 100: 0.7543, 300: 0.7537}
最优轮数 m=97, 对应测试集AUC=0.7544


## 10. GBDT 回归

前面 AdaBoost 做分类，现在进入 GBDT 回归。核心问题：回归场景里，下一棵树要学什么？

**结论：每一棵树不去直接猜房价，专门去拟合上一轮模型没猜对的差值——残差。**

先从最简单的起点开始：只用一个常数去预测所有房子单价。

- 平方误差（MSE）最小的常数，就是全部房价的平均值 $\bar{y}$；
- 初始模型：$F_0(x) = \bar{y}$，不管房子房龄多少，全部猜这个平均价；
- 残差定义：$r_i = y_i - F_0(x_i)$，真实房价减去平均房价，代表还差多少没猜对。

GBDT 的思路：第一棵树就去学这个残差；加上树的预测值之后，再算新残差，下一棵树继续学新残差，一轮一轮补齐误差。

例子：老师批改作业，先给全班一个基础分（平均分），然后记录每个人扣了多少分（残差）；第一棵树专门学“哪些学生扣得多”，补上这部分差值；补完再看剩下还差多少，交给下一棵树继续补。

### 10.1 案例

构造模拟的房龄—房价数据集，算出初始平均值、残差、训练 MSE、验证 MSE，作为 GBDT 回归的起点。

- 数据集：60 条训练样本、300 条验证样本；
- 规律：房价随房龄变大缓慢下降，叠加一点点正弦波动 + 高斯噪声，模拟真实房产数据；
- 只拿平均值预测，没有用到房龄任何信息，所以误差肯定不小，后面 GBDT 每一棵树就是用来降低这个 MSE。

In [ ]:
import numpy as np
# 固定随机种子
rng = np.random.default_rng(3)
# 生成60条训练样本：房龄，reshape成二维矩阵（sklearn要求特征是二维）
age = (rng.uniform(0,30,60)).reshape(-1,1)
# 真实房价公式：基础线性衰减 + 正弦波动（非线性）+ 高斯噪声
price = 3.2 - 0.06*age.ravel() + 0.25*np.sin(age.ravel()/3) + rng.normal(0,0.18,60)

# 生成300条验证集，用来评估泛化误差
rv = np.random.default_rng(33)
age_v = rv.uniform(0,30,300).reshape(-1,1)
price_v = 3.2 - 0.06*age_v.ravel() + 0.25*np.sin(age_v.ravel()/3) + rng.normal(0,0.18,300)

f0 = price.mean()
# 残差 = 真实房价 - 平均预测值，也就是F0没有猜对的差值
res = price - f0
# 打印前5条样本对应的残差（还差多少）
print(
    "他们的残差：", np.round(res[:5], 3).tolist()
    )
# 训练集MSE：残差平方的均值；验证集MSE：验证房价和f0的平方误差均值
print(
    "训练 MSE：", round(np.mean(res**2), 4), 
    "验证 MSE：", round(np.mean((price_v - f0)**2), 4)
    )


他们的残差： [0.974, 0.304, -0.333, -0.482, 1.02]
训练 MSE： 0.2532 验证 MSE： 0.3217


训练 MSE=0.2532，验证 MSE=0.3217：

- 验证误差比训练略高，正常，因为验证集是新样本；
- 这个 MSE 就是 GBDT 的起点，后续每加一棵树，目标就是把这个 MSE 往下压。

**注意：**

- AdaBoost 分类：样本错了就提高样本权重，下一棵树重点关注高权重样本；
- GBDT 回归：不用权重！下一棵树直接拟合残差（梯度），这是两者最核心的区别。

## 11. 第一棵残差树

得到初始模型 $F_0 = \bar{y}$（全部预测成平均房价），算出每一条样本的残差 $r_i = y_i - F_0$。

GBDT 最关键的一步：**第一棵树不去直接预测房价，专门拟合残差！**

1. 树的类型：`max_depth=1`，也就是树桩（只有一刀、两个叶子），非常简单，方便看明白分割点。
2. 回归树叶子规则：回归树叶子输出 = 落在这个叶子里所有样本**残差的平均值**（不是房价平均！）。
3. 这棵树学到的规律：
   - 房龄小（≤10.323）的新房：残差平均是正数 0.6049，意思是新房被初始平均值估低了，要往上补；
   - 房龄大（>10.323）的老房子：残差平均是负数 −0.3025，意思是老房子被初始平均值估高了，要往下扣。
4. 学习率 lr（步长）：不能一股脑把整棵树的结果全部加上。
   - lr=1：直接完整加上树的输出，步子跨很大；
   - lr=0.1：只加树输出的 1/10，慢慢走、更稳健，防止一步学过头、过拟合。
5. 公式：

$$F_1(x) = F_0(x) + \eta h_1(x)$$

$h_1(x)$ 就是这棵残差树的预测值。

例子：老师统一给所有人基础分（平均分），记录每个人还差多少分（残差）。现在这棵树发现：低年级（新房）普遍少加 0.61 分，高年级（老房）普遍多扣 0.3 分。学习率就是：这次先补一点点，不要一次性补满，避免矫枉过正。

In [12]:
# 导入sklearn回归决策树，用来训练深度为1的树桩
from sklearn.tree import DecisionTreeRegressor

# 新建回归树，max_depth=1 代表只能切一刀，两个叶子（树桩）
# 第一棵树不学目标 y（这里原本是房价 price），而是去拟合上一轮预测剩下的误差——残差res
t1 = DecisionTreeRegressor(max_depth=1).fit(age, res)

# 获取这一刀的分割阈值（切在哪)
cut = t1.tree_.threshold[0]
# 判断每个样本是不是落在左边：房龄 <= 分割点
# ravel()：将二维数组平摊为一维数组
left = age.ravel() <= cut
print("这一刀：房龄 <= ", round(cut, 3))

# left.sum() 左边样本数量；~left 取反就是右边样本；res[left].mean() 左边残差均值
print(f"左边 {left.sum()} 套，残差平均值 {res[left].mean():.4f}，右边 {(~left).sum()} 套，残差平均值 {res[(~left)].mean():.4f}")

# 尝试两组学习率：1.0 和 0.1
for lr in [1.0, 0.1]:
    # 更新模型：旧模型 + 学习率 * 残差树的预测:训练
    f1 = f0 + lr * t1.predict(age)
    # 更新模型：旧模型 + 学习率 * 残差树的预测:验证
    f1_v = f0 + lr * t1.predict(age_v)
    
    # 打印训练集MSE、验证集MSE
    print(f"学习率 {lr} : 训练MSE {np.mean((price - f1)**2):.4f}, 验证MSE {np.mean((price_v - f1_v)**2):.4f}")

这一刀：房龄 <=  10.323
左边 20 套，残差平均值 0.6049，右边 40 套，残差平均值 -0.3025
学习率 1.0 : 训练MSE 0.0702, 验证MSE 0.0868
学习率 0.1 : 训练MSE 0.2184, 验证MSE 0.2810


运行结果：

- 分割点：10.323，树自动找到最优一刀，用房龄 10.323 把房子分成新房组、老房组；
- 叶子输出：左边残差均值 0.6049，右边 −0.3025；
- 对照起点（训练 MSE=0.2532，验证 MSE=0.3217）：
  - lr=1.0：加上整棵树之后误差大幅下降，训练 0.0702，验证 0.0868，效果提升巨大；
  - lr=0.1：只加 10%，所以 MSE 下降很少（训练 0.2184，验证 0.2810），离最优还差很远，后面还需要继续加很多树。

学习率理解：

- 学习率大 → 收敛快，但是容易震荡、过拟合；
- 学习率小 → 需要更多树才能收敛，但是泛化更好、更稳；工业界 GBDT/XGBoost 一般用 0.01~0.1 的小 lr。

## 12. 手写一个 GBDT 回归

前面我们手动走了一轮：算残差 → 训练一棵小树 → 加上学习率更新预测。这个过程写成循环，就实现了最简版 GBDT 回归。

把“算 $F$ → 长一棵小树 → 走一小步”写成循环，就是完整的 GBDT 回归：

$$F_0(x) = \bar{y}, \qquad r_i = y_i - F_{m-1}(x_i), \qquad F_m(x) = F_{m-1}(x) + \eta h_m(x)$$

- $F_0$：初始预测，全部等于标签平均值；
- $r_i$：残差（当前模型还差多少没猜对）；
- $h_m(x)$：第 m 棵小树，专门拟合残差；
- $\eta$：学习率，每一步只走一小步。

循环很多轮，一棵树一棵树持续补误差。

In [13]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
# 导入sklearn官方GBDT回归器，用来和我们手写版本对照
from sklearn.ensemble import GradientBoostingRegressor

# 手写GBDT训练函数
# x：特征，y：标签，n_trees：一共多少棵树，lr：学习率，depth：每棵树最大深度
def gbdt_fit(x, y, n_trees, lr, depth):
    f0 = y.mean()               # F0：初始预测，所有样本统一为y的平均值
    F = np.full(len(y), f0)     # 初始化预测数组，一开始全部等于平均值
    trees = []                  # 用来保存训练出来的每一棵树，后面预测要用
    
    for m in range(n_trees):    # 循环，依次训练每一棵树
        r = y - F               # 计算残差 = 真实值 - 当前模型预测值
        # 训练一棵回归树，标签是残差r，不是原始y！
        t = DecisionTreeRegressor(max_depth=depth).fit(x, r)
        # 更新预测：旧预测 + 学习率 * 当前树对样本的预测
        F = F + lr * t.predict(x)
        trees.append(t)         # 把这棵树存进列表
        
    return f0, trees            # 返回初始均值 + 全部树，打包成model

# 手写GBDT预测函数
def gbdt_predict(model, x, lr):
    f0, trees = model           # 取出初始均值和所有树
    # 预测公式：初始均值 + 学习率 * 所有树预测结果相加
    return f0 + lr * sum(t.predict(x) for t in trees)

# 训练手写GBDT：100棵树，学习率0.1，每棵树深度1（树桩）
mine = gbdt_fit(age, price, 100, 0.1, 1)

# 验证集上评估手写版本
print("自己写的，验证 MSE：", round(np.mean((price_v - gbdt_predict(mine, age_v, 0.1)) ** 2), 4))

# ================= 补全官方对比部分 =================
# 训练官方GBDT：参数与手写版完全一致
official = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0)
official.fit(age, price)

# 验证集上评估官方版本
official_preds = official.predict(age_v)
print("官方库的，验证 MSE：", round(np.mean((price_v - official_preds) ** 2), 4))

自己写的，验证 MSE： 0.0441
官方库的，验证 MSE： 0.0441


在代码的基础上，添加 GBDT 模型做对照。

In [14]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
# 导入sklearn官方GBDT回归器
from sklearn.ensemble import GradientBoostingRegressor

# 手写GBDT训练函数
# x：特征，y：标签，n_trees：一共多少棵树，lr：学习率，depth：每棵树最大深度
def gbdt_fit(x, y, n_trees, lr, depth):
    # F0：初始预测，所有样本统一为y的平均值
    f0 = y.mean()               
    # 初始化预测数组，一开始全部等于平均值
    F = np.full(len(y), f0)     
    # 用来保存训练出来的每一棵树，后面预测要用
    trees = []                  
    
    # 循环，依次训练每一棵树
    for m in range(n_trees):    
        # 计算残差 = 真实值 - 当前模型预测值
        r = y - F               
        # 训练一棵回归树，标签是残差r，不是原始y！
        t = DecisionTreeRegressor(max_depth=depth).fit(x, r)
        # 更新预测：旧预测 + 学习率 * 当前树对样本的预测
        F = F + lr * t.predict(x)
        # 把这棵树存进列表
        trees.append(t)         
        
    # 返回初始均值 + 全部树，打包成model
    return f0, trees            

# 手写GBDT预测函数
def gbdt_predict(model, x, lr):
    # 取出初始均值和所有树
    f0, trees = model          
    # 预测公式：初始均值 + 学习率 * 所有树预测结果相加
    return f0 + lr * sum(t.predict(x) for t in trees)

# 训练手写GBDT：100棵树，学习率0.1，每棵树深度1（树桩）
mine = gbdt_fit(age, price, 100, 0.1, 1)

# 训练sklearn原生GBDT（参数保持与手写完全一致）
sk_gbdt = GradientBoostingRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=1, 
    random_state=0
).fit(age, price)

# 验证集上评估手写版本
print("自己写的，验证 MSE：", round(np.mean((price_v - gbdt_predict(mine, age_v, 0.1)) ** 2), 4))

# 验证集上评估sklearn官方版本
print("sklearn的，验证MSE:", round(np.mean((price_v - sk_gbdt.predict(age_v)) ** 2), 4))

# 计算两组预测结果最大差值（用来证明手写逻辑和官方完全一致）
print("两者的预测最多差：", float(np.abs(gbdt_predict(mine, age_v, 0.1) - sk_gbdt.predict(age_v)).max()))

自己写的，验证 MSE： 0.0441
sklearn的，验证MSE: 0.0441
两者的预测最多差： 1.3322676295501878e-15


自己写的 GBDT 与 sklearn 的 `GradientBoostingRegressor` 验证 MSE 都是 0.0441，两者预测最多只差约 $1.33 \times 10^{-15}$，说明手写循环和官方实现完全一致。

- AdaBoost：分类为主，每棵树改**样本权重**，重点关注难分样本；树权重不一样；
- GBDT：回归/分类都可以，每棵树拟合**残差（负梯度）**，不修改样本权重；统一用学习率缩放。

## 13. 学习率和树的数量

GBDT 里最重要的两个超参数：**学习率 lr** 和 **树的数量 n_estimators**。

学习率越大，步子迈得越大，很快摸到最低点，但继续加树就会过头、开始过拟合；学习率越小，步子小，走得慢，不容易冲过头，往往泛化更好，但是需要更多棵树才能到达最好效果。

这里用到一个关键 API：`staged_predict()`，它可以依次返回第 1 棵、第 2 棵、……、第 500 棵树累加之后的预测结果，不用训练 500 次模型，一次性拿到每一步迭代后的验证误差，画出误差曲线。

三个学习率：1.0、0.3、0.1，全部最多训练 500 棵树桩（`max_depth=1`），观察验证集 MSE 的变化。

**重点区分：验证集 MSE 先下降、降到最低点之后，继续增加树，MSE 反而上升 → 就是过拟合。**

类比理解：下山找谷底。

In [15]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# 目的：学习率 1.0、0.3、0.1，最多训练500棵树桩；跟踪验证MSE，找到MSE最低的位置
for lr in [1.0, 0.3, 0.1]:
    # 构建GBDT：最多500棵树，当前循环的学习率，每棵树是深度1的树桩，固定随机种子保证结果可复现
    m = GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=lr,
        random_state=0,
        max_depth=1
    ).fit(age, price)
    
    # staged_predict：依次返回累加1棵、2棵、…、500棵树后的预测值
    # 遍历每一步预测，计算验证集MSE，存入mse列表
    mse = [np.mean((price_v - p) ** 2) for p in m.staged_predict(age_v)]
    
    # np.argmin(mse)：找到MSE最小的下标，下标从0开始，所以+1才是树的编号
    # 依次打印：最低点位置、最小MSE；第10棵、第100棵、第500棵时的MSE
    print(f"学习率 {lr}: 第{int(np.argmin(mse))+1:>3}棵 最低{min(mse):.4f}; 10棵 {mse[9]:.4f}, 100棵 {mse[99]:.4f}, 500棵 {mse[499]:.4f}")

学习率 1.0: 第  7棵 最低0.0457; 10棵 0.0491, 100棵 0.0547, 500棵 0.0656
学习率 0.3: 第 19棵 最低0.0433; 10棵 0.0514, 100棵 0.0468, 500棵 0.0519
学习率 0.1: 第 78棵 最低0.0436; 10棵 0.1159, 100棵 0.0441, 500棵 0.0482


结果说明：

- lr=1.0：大步往下冲，很快踩到谷底（第 7 棵最低 0.0457），刹不住脚，继续往前走跑到另一边上坡（500 棵升到 0.0656，过拟合）；
- lr=0.3：中等步子，表现不错（第 19 棵最低 0.0433）；
- lr=0.1：小碎步慢慢挪，前期很慢（10 棵还有 0.1159），需要走很多步才能到谷底（第 78 棵最低 0.0436），不容易冲过头。

学习率大 → 收敛快，但是容易震荡、过拟合；学习率小 → 需要更多树才能收敛，但是泛化更好、更稳。

## 14. 早停

发现的问题：树不是越多越好，验证误差降到最低点之后，继续加树就会过拟合。人工盯着曲线找停止点很麻烦，早停（Early Stopping）就是让程序自动判断什么时候停下来。

原理：从训练集里面切一小块出来当内部验证集，每新增一棵树就看这个内部验证集的误差；如果连续 `n_iter_no_change` 轮，误差不再变小，就立刻停止训练，不再继续生成后面的树，防止过拟合。

**一个非常重要的规矩：**早停用的这份验证子集，不能是我们最开始单独分出来的测试集（那 300 套）。测试集全程只用来最后评估最终泛化能力，不能参与训练、不能用来决定什么时候停，否则就数据泄露、结果虚高。

参数说明：

- `n_iter_no_change`：耐心值，连续多少轮不下降就停机；
- `validation_fraction`：从原始训练集里划出多大比例，作为早停的内部验证集，sklearn 默认 0.1；
- 这里学习率 0.1、树桩深度 1、上限最多 500 棵树，交给早停去决定真实树数量。

In [16]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# 目的：从训练集划出一部分做内部验证集；验证误差连续10轮不下降就提前停止，观察最终训练了多少棵树
# validation_fraction默认0.1，自动切训练集的10%作为早停用的内部验证集
es = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=1,
    # n_iter_no_change=10：耐心=10，连续10轮内部验证MSE没有下降，就停止
    n_iter_no_change=10,
    random_state=0
).fit(age, price)

# es.n_estimators_：模型实际训练出来的树的数量（早停触发后会小于500）
print("实际长了", es.n_estimators_, "棵树")

# 在【独立的外部验证集（300套，早停全程没见过这份数据）】计算MSE
print("验证集 MSE：", round(np.mean((price_v - es.predict(age_v))**2), 4))

实际长了 58 棵树
验证集 MSE： 0.0465


输出说明：上限 500 棵，早停在第 58 棵停止；在独立的 300 套外部验证集上，MSE 为 0.0465。

类比：爬山找山谷谷底。你边走边看旁边的标尺（内部验证误差），连续走 10 步都没有更低，就判定已经到谷底，不再往前走，避免爬到对面山坡。

这个用来判断停不停的标尺，本身样本很少、带有随机噪声，所以它找到的停止点不一定是全局最优最低点，数据量越大噪声越小，早停越准。

## 15. 负梯度

**残差就是负梯度。**

前面我们一直说 GBDT 在拟合残差，这只是平方损失下的特例。

梯度下降的通用思想：沿着损失函数的负梯度方向，走一小步，降低损失。

平方损失：

$$L = \frac{1}{2}\left(y_i - F(x_i)\right)^2$$

对当前预测 $F(x_i)$ 求导：

$$\frac{\partial L}{\partial F(x_i)} = -\left(y_i - F(x_i)\right)$$

负梯度：

$$-\frac{\partial L}{\partial F(x_i)} = y_i - F(x_i) = r_i$$

平方损失时，负梯度刚好 = 残差。

关键点：

- 普通梯度下降：直接修改每个样本的预测值 $F(x_i)$；
- GBDT：不能直接改（直接改只会死记训练集，泛化差），训练一棵树来近似这个负梯度，树可以对新的、没见过的 x 输出更新量；学习率 $\eta$ 就是梯度下降的步长。

GBDT 本质就是函数空间上的梯度下降；每棵树拟合的目标，就是损失函数的负梯度（叫伪残差），所以全称叫梯度提升 Gradient Boosting。

平方损失只是刚好伪残差 = 普通残差；换别的损失，拟合目标就不再是简单残差了。

## 16. 分类时，残差是什么？

GBDT 二分类里，拟合的伪残差 = **真实标签 y − 当前预测概率 p**，不是简单的 0/1 残差！

GBDT 做二分类，和逻辑回归思路一致：

- 模型输出 F：对数几率（logit），不是概率；
- Sigmoid 把对数几率转成概率：$p = \sigma(F) = \frac{1}{1 + e^{-F}}$；
- 损失函数：交叉熵损失；
- 对 F 求导算出负梯度（伪残差）：$-\frac{\partial L}{\partial F} = y - p$。

其中：

- y：真实标签，取值 0 或 1；
- p：当前模型预测为正类的概率。

对比回归：回归平方损失负梯度是 (y−F)；二分类交叉熵的负梯度是 (y−p)。它依然是伪残差，只是这里的“预测值”是概率，不是回归的连续输出。

例子：20 个同学，11 人通过考试：

- $p_0 = 11/20 = 0.55$；
- 正样本（通过，y=1）：$y-p=1-0.55=0.45$；
- 负样本（不通过，y=0）：$y-p=0-0.55=-0.55$。

这就是分类 GBDT 第一轮要拟合的目标！分类这里的残差不是 y−0.5，是 y−当前预测概率 p；每一轮 p 都会随模型更新而变化，伪残差也跟着变。

### 16.1 案例

复用前面 20 位同学数据集（复习时长、是否通过考试），演示 GBDT 二分类初始化：从全局通过率的对数几率出发，计算第一轮伪残差 y−p。

In [17]:
import numpy as np
import pandas as pd

# 复习时长
hours = np.array([1, 2, 2.5, 3, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 9, 10, 11, 12, 13, 14, 15])
# passed:标签，1：通过 0：没通过
passed = np.array([0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1])

# 全局先验通过率（全部样本里通过的比例）
p0 = passed.mean()
# F0：初始对数几率 log( p / (1-p) )，GBDT分类模型的初始F值
F0 = np.log(p0 / (1 - p0))

print(f"通过 {passed.sum()}人，通过率 p0={p0}，对数几率 F0=ln({p0}/{1-p0:.2f})={F0:.4f} ")

# 计算第一轮的伪残差：y - p0，也就是交叉熵损失下的负梯度
r1 = passed - p0
print("每个人的残差 y-p:", np.round(r1, 2).tolist())

通过 11人，通过率 p0=0.55，对数几率 F0=ln(0.55/0.45)=0.2007 
每个人的残差 y-p: [-0.55, -0.55, -0.55, -0.55, -0.55, -0.55, 0.45, -0.55, -0.55, 0.45, 0.45, -0.55, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45]


通过 11 人，通过率 p0=0.55，对数几率 F0=ln(0.55/0.45)=0.2007。

这一组 −0.55 / 0.45 就是第一棵回归树要去拟合的目标（伪残差）。训练完这棵树后，乘以学习率加到 F 上，更新 p，下一轮再重新计算新的 (y−p)，循环迭代。

## 17. 第一棵树：叶子输出不是残差的平均值

之前学 GBDT 回归：损失是均方误差 MSE，叶子里面直接输出残差平均值，最简单。

现在是信贷违约 / 二分类，损失是交叉熵。交叉熵不是二次函数，没有“取残差平均”这么简单的最优解。

牛顿法的思想：在当前这一步，用泰勒展开近似损失，走一小步找到最好的更新量 γ：

- 分子：$\sum (y_i - p_i)$，一阶导数（梯度，残差就是交叉熵损失的负梯度），代表损失往哪个方向下降；
- 分母：$\sum p_i(1-p_i)$，二阶导数（海森），代表损失曲面弯得有多陡；
- $\gamma = \frac{一阶和}{二阶和}$：这就是这个叶子节点，对数几率（logit）上面要加上的最优增量。

例子：想象下山（最小化损失）。MSE 是光滑的抛物线山坡，每一块平地最优步子就是残差平均；交叉熵这个山坡不是抛物线，不能直接用残差平均当步子。牛顿法不仅看现在往哪走（一阶），还看地面有多陡（二阶），把步子修正一下，就是 γ。

- 回归 MSE：叶子输出 = 残差均值；
- 二分类交叉熵 GBDT：叶子输出 $\gamma = \frac{残差之和}{\sum p(1-p)}$。

我们 GBDT 分类，所有树都是在对数几率 F（logit）上面叠加更新量，不是直接改概率 p！最后再用 sigmoid 转成概率。还有学习率（步长 0.1），防止一步跨太大冲过头过拟合。

In [18]:
# 导入回归决策树，这里用max_depth=1就是【树桩，只有一刀，两个叶子】
from sklearn.tree import DecisionTreeRegressor
import numpy as np

# 1. 训练一个深度=1的树桩，表面上拟合残差r1
hours = hours.reshape(-1, 1)
t1c = DecisionTreeRegressor(max_depth=1).fit(hours, r1)

# 2. 拿到这棵树桩的分割阈值
cut = t1c.tree_.threshold[0]

# 3. L是布尔掩码：判断每个样本是不是【复习时长 ≤ 切割阈值cut】，也就是左叶子样本
L = hours.ravel() <= cut

# 4. 初始化数组p，20个样本全部填充初始概率p0
p = np.full(20, p0)

# 5. 核心计算！字典gam存左右两个叶子的γ
# 循环两个分支：左分支L、右分支~L（~代表取反，大于cut的样本）
# m是当前叶子的样本掩码；分子：当前叶子里面所有样本残差求和；分母：当前叶子p*(1-p)求和
gam = {side: r1[m].sum() / (p[m] * (1 - p[m])).sum() for side, m in [('左', L), ('右', ~L)]}

# 6. 打印分割点
print("这一刀：复习时长：", cut)
# 7. 遍历左右两个叶子，打印对比：γ 和【残差的平均值】
for side, m in [('左', L), ('右', ~L)]:
    print(f"{side}边 {m.sum()} 人;残差之和{r1[m].sum():.2f},p(1-p) 之和 {(p[m] * (1 - p[m])).sum():.4f},叶子输出 {gam[side]:.4f}(残差平均值 {r1[m].mean():.4f})")

# 8. 更新对数几率F：F = 旧F0 + 学习率 * γ；左叶子加左γ，右叶子加右γ
F1 = F0 + 0.1 * np.where(L, gam['左'], gam['右'])

# 9. 打印更新之后，左右叶子对应的新logit值
print("学习率0.1，新的对数几率（左右）：", round(F0 + 0.1 * gam['左'], 4), round(F0 + 0.1 * gam['右'], 4))
# 10. sigmoid：1/(1+exp(-F))，把新的对数几率转回概率，看挂科概率怎么变化
print("新的概率（左、右）：", round(1 / (1 + np.exp(-(F0 + 0.1 * gam["左"]))), 4), round(1 / (1 + np.exp(-(F0 + 0.1 * gam["右"]))), 4))

这一刀：复习时长： 6.25
左边 9 人;残差之和-3.95,p(1-p) 之和 2.2275,叶子输出 -1.7733(残差平均值 -0.4389)
右边 11 人;残差之和3.95,p(1-p) 之和 2.7225,叶子输出 1.4509(残差平均值 0.3591)
学习率0.1，新的对数几率（左右）： 0.0233 0.3458
新的概率（左、右）： 0.5058 0.5856


运行结果：

- 分割点：复习时长 6.25；
- 左边 9 人：残差之和 −3.95，p(1−p) 之和 2.2275，叶子输出 −1.7733（残差平均值 −0.4389）；
- 右边 11 人：残差之和 3.95，p(1−p) 之和 2.7225，叶子输出 1.4509（残差平均值 0.3591）；
- 学习率 0.1 更新后，左右对数几率分别为 0.0233、0.3458。

logit 只是模型内部干活的数，业务上我们关心违约 / 挂科概率，所以用 sigmoid 转回 0~1 之间的概率。例子里：复习小于 6.25h，挂科概率 50.58%；大于 6.25h，挂科概率 58.56%，符合直觉。

## 18. 引入 sklearn 中的内容

刚才手工推了单棵树桩版本的 GBDT 分类：叶子不是残差平均，而是「一阶和 ÷ 二阶和」的牛顿一步近似。

现在把这个逻辑封装成循环，连续训练 50 棵树（50 轮提升），然后和 sklearn 官方的 `GradientBoostingClassifier` 做对照。sklearn 里面二分类 GBDT 底层，就是这套逻辑。

类比：我们自己按照 Friedman 原始论文的规则，一步一步手工复刻“爬山下山”；sklearn 是工业封装好的成品登山机。如果代码逻辑一致，两者算出的对数几率（`decision_function` 输出）几乎一模一样。

补充：`decision_function()` 返回的就是对数几率 F，和我们函数最后 return 的 F 是同一个东西；再套 sigmoid 才是概率 `predict_proba`。

In [19]:
# 导入sklearn官方的GBDT分类器
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier

# =========手写的GBDT==========
# 自定义函数：复刻Friedman原版二分类GBDT
def gbdt_clf(x, y, n_trees, lr, depth):
    # 【初始化对数几率F】
    # 起点：用样本正负类比例，也就是log(正样本数/负样本数)，sklearn也是这个初始化！
    F = np.full(len(y), np.log(y.mean() / (1 - y.mean())))

    # 循环：一轮一轮训练每一棵树，一共n_trees轮提升
    for m in range(n_trees):
        # 用当前的F，sigmoid转换成预测概率p
        p = 1 / (1 + np.exp(-F))
        # 计算伪残差 r = y-p （交叉熵损失的负梯度，一阶导数）
        r = y - p
        # 训练一棵回归树，用特征x去拟合伪残差r，这棵树用来找怎么切分样本
        t = DecisionTreeRegressor(max_depth=depth).fit(x, r)
        # apply(x)：得到每个样本落在这棵树的哪一片叶子上，返回叶子编号
        leaf = t.apply(x)

        # 遍历这棵树里面所有不重复的叶子编号，一片一片叶子单独计算γ
        for k in np.unique(leaf):
            # 找到落在第k号叶子里的全部样本
            mask = (leaf == k)
            # 核心公式：γ = 叶子内残差之和 / 叶子内 p*(1-p)之和（一阶/二阶，牛顿一步）
            gamma = r[mask].sum() / (p[mask] * (1 - p[mask])).sum()
            # 更新对数几率F：旧F + 学习率 × γ
            # 只更新落在这片叶子里面的样本！不同叶子加不同的增量
            F[leaf == k] += lr * gamma
    # 循环全部树训练完成，返回最终所有样本的对数几率F
    return F

#调用我们自己写的GBDT，50棵树，学习率0.1，每棵都是深度1的树桩
F_mine = gbdt_clf(hours, passed, 50, 0.1, 1)

# =======引入sklearn官方的GradientBoostingClassifier======
#调用官方GBDT，50棵树，学习率0.1，每棵都是深度1的树桩
skc = GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=1,
    random_state=0
).fit(hours, passed)

# 打印对比：前5个样本，我们手写版本算出来的对数几率
print("自己手写的，前5人的对数几率：", np.round(F_mine[:5], 4).tolist())
# sklearn的decision_function输出，就是对数几率F
print("sklearn的，前5人的对数几率：", np.round(skc.decision_function(hours)[:5], 4).tolist())
# 计算全部20个样本上，两个版本F的最大差值
print("20 个人最多差：", float(np.abs(F_mine - skc.decision_function(hours)).max()))

自己手写的，前5人的对数几率： [-2.9233, -2.9233, -2.9233, -2.9233, -2.9233]
sklearn的，前5人的对数几率： [-2.9233, -2.9233, -2.9233, -2.9233, -2.9233]
20 个人最多差： 8.881784197001252e-16


自己手写的和 sklearn 的前 5 人对数几率完全一致，20 个人最多只差约 $8.88 \times 10^{-16}$，说明手写循环复刻了 sklearn 的二分类 GBDT 逻辑。

## 19. 信贷数据：树越多越好吗？

前面一直以为：GBDT 树越多，模型就越厉害。这一节拿信贷违约预测数据做实验，回答标题的问题：树越多越好吗？

例子：好比背一套信贷案例题库（训练集）。

- 刚开始背，背得越多（树增加），你对规律理解越深，考试（测试集，新客户）分数越来越高；
- 背到一定程度之后，你开始死记硬背题目里的错别字、偶然的巧合噪声，而不是通用的判断逻辑；
- 这时候继续背更多题（继续加树），你在真实新客户上的判断能力反而下降——这就是过拟合。

配套知识点：`staged_predict_proba` 是 sklearn GBDT 专门的接口：它可以逐步拿到第 1 棵、第 2 棵……一直到第 500 棵树叠加完之后的预测概率，不用反复训练 500 次模型，非常方便观察 AUC 变化曲线。

In [20]:
# 导入对应的包
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('../data/信贷数据.csv')

# 分开定义：数值型特征、类别型特征
num = ["年龄", "月收入", "负债收入比", "额度使用率", "逾期次数", "工作年限"]
cat = ["学历", "住房", "职业"]

# 划分训练集、测试集，测试集占20%
X_train, X_test, y_train, y_test = train_test_split(df[num+cat], df['是否违约'].to_numpy(), test_size=0.2, stratify=df['是否违约'], random_state=42)
# 预处理流水线：数值列原样保留（passthrough）；分类列做OneHot独热编码
enc = ColumnTransformer([
    ('num', "passthrough", num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
]).fit(X_train)

# 对训练、测试数据执行编码转换
E_train, E_test = enc.transform(X_train), enc.transform(X_test)

# 训练GBDT模型，一次性最多准备500棵树；学习率0.1，每棵树最大深度3
gb = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=3,
    random_state=0
).fit(E_train, y_train)

# staged_predict_proba：生成一个迭代器，依次返回【叠加前1棵树、前2棵、...前500棵树】的预测概率
# p[:,1] 取第二列，是预测【违约=1】的概率
# 循环每一步，计算对应的测试集AUC，存入auc列表
auc = [roc_auc_score(y_test, p[:, 1]) for p in gb.staged_predict_proba(E_test)]
# 挑选几个有代表性的树数量：1、10、50、100、200、500，打印对应的测试集AUC
print("测试集 AUC：", {m: round(auc[m-1], 4) for m in [1, 10, 50, 100, 200, 500]})
print("最高：第", int(np.argmax(auc)) + 1, "棵，", round(max(auc), 4))

测试集 AUC： {1: 0.6781, 10: 0.7291, 50: 0.7555, 100: 0.7514, 200: 0.7429, 500: 0.7207}
最高：第 58 棵， 0.7574


测试集 AUC 随树数量的变化：

| 轮数 | 1 | 10 | 50 | 100 | 200 | 500 |
| --- | --- | --- | --- | --- | --- | --- |
| AUC | 0.6781 | 0.7291 | 0.7555 | 0.7514 | 0.7429 | 0.7207 |

最高在第 58 棵，AUC 0.7574。树太少学得不够，树太多开始过拟合，AUC 先升后降。

## 20. 学习率和早停

刚才发现：树加多了会过拟合。所以引出两个核心工具：学习率、早停。

用爬山的比喻：

- 学习率就是你每一步跨出去的步子大小：lr=0.1 是大步，lr=0.02 是小碎步；
- 早停就是爬山时的刹车机制：专门留出一小块“验证山坡”，如果连续 20 步往上走，高度都不再提升，就判定已经到顶，立刻停下，不再继续爬，防止翻过山顶往下坡走（过拟合）。

两个很关键的规律：

- 步子大（lr 大），上升快，很快就到最高点，用很少的树就停下来；缺点是容易一脚迈过最高点；
- 步子小（lr 小），走得慢，需要很多很多小步（很多树）才能爬到最高点；好处是定位最高点更精细，往往泛化效果更好。

规则说明：

- `validation_fraction=0.2`：从训练集里面，单独切出 20% 当验证集，验证集≠测试集！验证集用来监控什么时候该刹车（早停）；测试集从头到尾只用来最后看最终效果，全程不参与选参数、不参与早停判断，符合机器学习规范，没有偷看考卷；
- `n_iter_no_change=20`：连续 20 轮验证集指标没有提升，就触发早停。

In [21]:
# 导入对应的包
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# 循环遍历三组不同的学习率：0.1、0.05、0.02，对比实验
for lr in [0.1, 0.05, 0.02]:
    # 初始化GBDT分类器
    m = GradientBoostingClassifier(
        n_estimators=2000,
        learning_rate=lr,
        max_depth=3,
        n_iter_no_change=20,
        validation_fraction=0.2,
        random_state=0
    ).fit(E_train, y_train)
    # 打印结果：对齐排版，输出当前学习率、实际训练出来的树数量、测试集AUC
    print(f"学习率 {lr:<4}: 长了 {m.n_estimators_:>4}棵，测试集 AUC {roc_auc_score(y_test, m.predict_proba(E_test)[:, 1])}")

学习率 0.1 : 长了   39棵，测试集 AUC 0.7550021084872159
学习率 0.05: 长了   83棵，测试集 AUC 0.7575489390980115
学习率 0.02: 长了  186棵，测试集 AUC 0.7542641379616477


学习率 0.1 / 0.05 / 0.02 分别训练 39 / 83 / 186 棵，测试集 AUC 分别为 0.7550 / 0.7575 / 0.7543。

## 21. 树的深度和子采样

已经学了学习率、早停，现在看 GBDT 另外两个非常核心的正则参数：`max_depth`（单棵树深度）和 `subsample`（子采样）。

两个参数的解读：

- `max_depth` 树的深度：每一棵树能想多深。深度 1 就是只问一个问题（树桩），判断能力很简单，不容易记噪声；深度 5 就是可以连环问 5 层问题，单棵树变得很聪明，但也很容易钻牛角尖，记住训练集里偶然的特例，容易过拟合。
- `subsample` 子采样：Friedman 提出的随机梯度提升，和袋装（bagging）思路很像。`subsample=0.8` 代表：训练每一棵新树的时候，随机抽 80% 的训练样本去训练这棵树，剩下 20% 样本不给这棵树看；`subsample=1.0` 就是全部样本都用。

好处两点：

1. 不同树看到的数据不一样，树之间的“想法”有差异，叠加之后方差更低，不容易一起记错噪声，抑制过拟合；
2. 每棵树只用一部分样本，计算更快。

注意：这里不再用测试集挑参数，改用 5 折分层交叉验证，只用训练集内部评估，符合规范，不偷看测试集答案。分层 5 折 `StratifiedKFold`：保证每一折里面，违约、不违约客户的比例和原始训练集保持一致，信贷这种不平衡数据集很适合。

In [22]:
# 导入交叉验证工具、分层折工具
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier

# 创建5折分层交叉验证器
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# 外层循环：遍历不同的树最大深度：1、2、3、5
for depth in [1, 2, 3, 5]:
    # 内层循环：遍历两种采样方案：1.0用全部样本、0.8只随机抽80%样本训练单棵树
    for sub in [1.0, 0.8]:
        # 构建GBDT模型
        m = GradientBoostingClassifier(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=depth,
            # 当前循环的子采样比例
            subsample=sub,
            n_iter_no_change=20,
            validation_fraction=0.2,
            random_state=0
        )
        # cross_val_score：5折交叉验证，每折计算AUC，返回5个AUC值，.mean()取平均值
        auc_cv_mean = cross_val_score(m, E_train, y_train, cv=cv, scoring='roc_auc').mean()
        # 打印：树深度、采样比例、5折平均AUC
        print(f"深度 {depth}, subsample {sub} :5折交叉验证 AUC {auc_cv_mean:.4f}")

深度 1, subsample 1.0 :5折交叉验证 AUC 0.7481
深度 1, subsample 0.8 :5折交叉验证 AUC 0.7480
深度 2, subsample 1.0 :5折交叉验证 AUC 0.7442
深度 2, subsample 0.8 :5折交叉验证 AUC 0.7459
深度 3, subsample 1.0 :5折交叉验证 AUC 0.7410
深度 3, subsample 0.8 :5折交叉验证 AUC 0.7404
深度 5, subsample 1.0 :5折交叉验证 AUC 0.7268
深度 5, subsample 0.8 :5折交叉验证 AUC 0.7308


5 折交叉验证平均 AUC：

| 深度 | subsample=1.0 | subsample=0.8 |
| --- | --- | --- |
| 1 | 0.7481 | 0.7480 |
| 2 | 0.7442 | 0.7459 |
| 3 | 0.7410 | 0.7404 |
| 5 | 0.7268 | 0.7308 |

总结：

1. `max_depth` 控制单棵树复杂度：太小容易欠拟合，太大容易过拟合；GBDT 一般不会用很深的树，常见 3~6。
2. `subsample` 是行采样（对样本采样，不是特征），随机梯度提升 SGB 的标志性设置，属于正则化手段。
3. 交叉验证用来在训练集内部对比多组超参数，选出最优组合之后，才最后在测试集上评估一次。
4. 对比前面袋装：袋装是每棵树独立训练、最后平均；GBDT 是串行依次建树，`subsample` 只是给每棵树抽一部分样本，不是独立训练，二者有区别。

## 22. 和之前模型进行对比

用完全一样的信贷数据、一模一样的训练集 / 测试集划分，把前面所有讲过的模型放在同台 PK：逻辑回归、随机森林、AdaBoost、GBDT。

对比：

- 逻辑回归：线性模型，相当于只看特征简单加权打分，规则很简单、解释性强，但是学不到特征之间复杂的交叉关系；
- 随机森林：并行的袋装树，很多独立大树一起投票，靠多样性来降低过拟合；
- AdaBoost：串行训练树桩（深度 1 的小树），重点去学前面分错的样本，给难样本加大权重；
- GBDT：串行拟合残差，每一棵树专门弥补上一轮模型的误差，是这几个里面经常效果上限最高的，但调参也更讲究。

重点：全部只用测试集 AUC 对比好坏，测试集全程没有用来选参数，只做最后的公平大比拼，保证比赛公平。

In [23]:
# 导入流水线工具，用来把预处理和模型打包在一起
from sklearn.pipeline import Pipeline
# 标准化工具，逻辑回归对量纲敏感，需要标准化数值特征
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# 导入逻辑回归模型
from sklearn.linear_model import LogisticRegression
# 导入随机森林、GBDT、AdaBoost
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import pandas as pd

df = pd.read_csv('../data/信贷数据.csv')

# 分开定义：数值型特征、类别型特征
num = ["年龄", "月收入", "负债收入比", "额度使用率", "逾期次数", "工作年限"]
cat = ["学历", "住房", "职业"]

# 划分训练集、测试集，测试集占20%
X_train, X_test, y_train, y_test = train_test_split(df[num+cat], df['是否违约'].to_numpy(), test_size=0.2, stratify=df['是否违约'], random_state=42)

# 预处理流水线：数值列原样保留（passthrough）；分类列做OneHot独热编码
enc = ColumnTransformer([
    ('num', "passthrough", num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
]).fit(X_train)

# 对训练、测试数据执行编码转换
E_train, E_test = enc.transform(X_train), enc.transform(X_test)

# 训练GBDT
gbdt = GradientBoostingClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.8,
    n_iter_no_change=20,
    validation_fraction=0.2,
    random_state=0
).fit(E_train, y_train)

# 逻辑回归需要单独流水线：数值特征标准化、类别特征独热编码
# 这条流水线接收原始 DataFrame，所以用 X_train / X_test
lr_model = Pipeline([
    ("pre", ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(drop='first', handle_unknown='ignore'), cat)
    ])),
    # 逻辑回归，调高迭代次数保证收敛
    ("lr", LogisticRegression(max_iter=1000))
]).fit(X_train, y_train)

# 随机森林
rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=20,
    random_state=0
).fit(E_train, y_train)

# AdaBoost
ada_c = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=300,
    random_state=0
).fit(E_train, y_train)

# 构建字典：名字对应模型输出的违约概率
models = {
    "逻辑回归": lr_model.predict_proba(X_test)[:, 1],
    "随机森林": rf.predict_proba(E_test)[:, 1],
    "AdaBoost": ada_c.predict_proba(E_test)[:, 1],
    "GBDT": gbdt.predict_proba(E_test)[:, 1],
}

# 循环遍历所有模型，逐个打印测试集AUC
for name, p in models.items():
    print(f"{name:<10} 测试集 AUC {roc_auc_score(y_test, p):.4f}")

逻辑回归       测试集 AUC 0.7711
随机森林       测试集 AUC 0.7557
AdaBoost   测试集 AUC 0.7537
GBDT       测试集 AUC 0.7609


测试集 AUC：

| 模型 | 测试集 AUC |
| --- | --- |
| 逻辑回归 | 0.7711 |
| 随机森林 | 0.7557 |
| AdaBoost | 0.7537 |
| GBDT | 0.7609 |

四个模型使用同一份训练集 / 测试集划分，测试集只在最后评估一次。